# Phase D pilot — 20 examples per condition (60 total)

Generates pilot data via OpenRouter / claude-sonnet-4-5, validates with regex leak checks + Qwen3-tokenizer token counts, and prints 5 random examples per condition for review.

Stops here. After the user OKs the samples, the same `gen.generate` call with `--n 500` (no `--pilot`) does the full sweep.

In [ ]:
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    !pip install -q localrouter pydantic transformers
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    os.environ['OPENROUTER_API_KEY'] = userdata.get('clr_openrouter')
    os.environ['HF_TOKEN'] = userdata.get('hf_token')
    REPO = '/content/drive/MyDrive/clr-worktest'
    if not os.path.isdir(REPO):
        !git clone https://github.com/Junekhunter/clr-worktest.git $REPO
    %cd $REPO
    !git pull
else:
    %cd /home/hunter/ai/clr_worktest

In [ ]:
# Clear any prior pilot files so we don't refuse-to-overwrite mid-run.
from pathlib import Path
for c in ['demos','first_person','sdf']:
    p = Path('data/pilot') / f'{c}.jsonl'
    if p.exists(): p.unlink()
Path('data/pilot').mkdir(parents=True, exist_ok=True)

In [ ]:
# Generate (each ~30-60s, 20 examples, sem(20))
!python -m gen.generate --condition demos        --pilot --n 20
!python -m gen.generate --condition first_person --pilot --n 20
!python -m gen.generate --condition sdf          --pilot --n 20

In [ ]:
# Validate (regex leak + token count + coverage)
!python -m gen.validate --pilot

In [ ]:
# Show 5 random examples per condition for eyeballing.
import json, random
from pathlib import Path
rng = random.Random(0)
for cond in ['demos','first_person','sdf']:
    p = Path(f'data/pilot/{cond}.jsonl')
    rows = [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
    print(f'\n{"="*80}\n{cond.upper()}  ({len(rows)} rows)\n{"="*80}')
    for r in rng.sample(rows, min(5, len(rows))):
        print(f'\n--- {r.get("_cell",{})}')
        print(f'USER: {r["user"]}')
        print(f'ASSISTANT: {r["assistant"]}')
        leaks = {k: v for k, v in r.items() if k.startswith("leak_")}
        if any(leaks.values()):
            print(f'!! self-reported leaks: {leaks}')